In [1]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [2]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [3]:
import pandas as pd
import numpy as np

In [4]:
# custom
from utils import *
import _run_constants as rc

# LOAD DATA

In [5]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [6]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


## EXAMPLES OF BYTE COMPARISONS

In [7]:
w1 = 'abhor'
w2 = 'cleft'
w3 = 'frown'
w1b = byte_encode_words(w1)
w2b = byte_encode_words(w2)
w3b = byte_encode_words(w3)

In [8]:
# no letters in common
w1b & w2b

0

In [9]:
# letters in common
w1b & w3b

147456

In [10]:
# bitwise or
w1b | w2b

673975

In [11]:
# this is the same as directly above
byte_encode_words('abhorcleft')

673975

# BUILD LEVEL 2

In [12]:
l2_list = np.full(shape = (10000000, 3), fill_value = -1, dtype = np.int32)
row_index = 0
for w1_be, w2_be in combinations(word_byte_list, 2):
    if w1_be & w2_be == 0:   
        # they share no letters in common
        l2 = w1_be | w2_be                          
        
        l2_list[row_index, :] = np.array([w1_be, w2_be, l2], dtype = np.int32)
        row_index += 1

# trim the data frame
l2_list = l2_list[:row_index, :]
print(l2_list.shape)
l2_df = pd.DataFrame(data = l2_list, columns = ['w1b', 'w2b', 'l2'])

(3213696, 3)


In [13]:
l2_df = l2_df.drop_duplicates(subset = 'l2').reset_index(drop = True)

In [14]:
l2_array = l2_df['l2'].to_numpy(dtype = np.int32)

# CREATE ARRAYS OF THE SAME SHAPE

In [ ]:
N_SAMPLE_SIZE = 10000
#N_SAMPLE_SIZE = tl2_array.shape[0]
l2_cut_range = range(0, l2_array.shape[0] + N_SAMPLE_SIZE, N_SAMPLE_SIZE)

# number of words/word bytes
n_wba = word_byte_array.shape[0]

# this are the word as bytes array
r_wba = np.repeat(a = np.reshape(word_byte_array, shape = (1, n_wba)), repeats = N_SAMPLE_SIZE, axis = 0)

l5_output = []

for i_l2_idx, l2_idx in enumerate(l2_cut_range[1:]):
    pre_l2_idx = l2_cut_range[i_l2_idx]
    print('l2', l2_idx)
    
    # sample some rows!
    # work with N_SAMPLE_SIZE
    tl2 = l2_array[pre_l2_idx:l2_idx]     

    # I want each value to be repeated 5977 times across. In the x direction
    # these are the l2 combos
    r_tl2t = np.reshape(np.repeat(a = tl2, repeats = n_wba), shape = (N_SAMPLE_SIZE, n_wba))
    
    # # compare two columns
    # (r_tl2t[:, 1] == r_tl2t[:, 100]).all()
    # # compare two rows
    # (r_wba[0, :] == r_wba[10, :]).all()

    # bitwise AND, check when the values are 0, meaning there are shared letters, and then convert the booleans to integers
    l2_and = ((r_tl2t & r_wba) == 0).astype(np.int32)
    # bitwise OR, multiply by the 1/0 matrix. This will indicate groups of three words that do not share a common letter.
    l2_or = (r_tl2t | r_wba) * l2_and
    
    # get the indices...
    i, j = np.where(l2_or > 0)
    # these are the indices of the word as bytes
    
    
    # we can also use the i,j args in the r_tl2t and r_wba matrices
    # l2_or is the level 3 word group
    # how do put this into a new matrix?
    l3_array = l2_or[i, j]
    
    # bop that into the same shape
    # I want each value to be repeated 5977 times across. In the x direction
    l3_cut_range = range(0, l3_array.shape[0] + N_SAMPLE_SIZE, N_SAMPLE_SIZE)

    for i_l3_idx, l3_idx in enumerate(l3_cut_range[1:]):
        pre_l3_idx = l3_cut_range[i_l3_idx]

        print('l3', l3_idx)
    
        # sample some rows!
        # work with N_SAMPLE_SIZE
        tl3 = l3_array[pre_l3_idx:l3_idx]     
        
        r_tl3t = np.reshape(np.repeat(a = tl3, repeats = n_wba), shape = (N_SAMPLE_SIZE, n_wba))
    
        l3_and = ((r_tl3t & r_wba) == 0).astype(np.int32)
        l3_or = (r_tl3t | r_wba) * l3_and
        # get the indices...
        i, j = np.where(l3_or > 0)
        
        # this is l4        
        l4_array = l3_or[i, j]

    
        # bop that into the same shape
        # I want each value to be repeated 5977 times across. In the x direction
        l4_cut_range = range(0, l4_array.shape[0] + N_SAMPLE_SIZE, N_SAMPLE_SIZE)

        for i_l4_idx, l4_idx in enumerate(l4_cut_range[1:]):
            pre_l4_idx = l4_cut_range[i_l4_idx]

            print('l4', l4_idx)
    
            # sample some rows!
            # work with N_SAMPLE_SIZE
            tl4 = l4_array[pre_l4_idx:l4_idx]     
        

            r_tl4t = np.reshape(np.repeat(a = tl4, repeats = n_wba), shape = (tl4.shape[0], n_wba))
            r_wba_l4 = np.repeat(a = np.reshape(word_byte_array, shape = (1, n_wba)), repeats = tl4.shape[0], axis = 0)
            l4_and = ((r_tl4t & r_wba_l4) == 0).astype(np.int32)

            l4_or = (r_tl4t | r_wba_l4) * l4_and
            i, j = np.where(l4_or > 0)

            l5_array = l4_or[i, j]

            l5_output.append(l5_array)


10000


KeyboardInterrupt: 

In [ ]:
mike_byte = byte_encode_words('mike')
babb_byte = byte_encode_words('babb')
mike_byte & babb_byte
mike_byte | babb_byte

In [ ]:
j

In [ ]:
# this still isn't making sense... the problem with this approach is that I need to be able to go back to words
# at some point. I'm not sure if I can at this point. Without having another big dumb matrix of words to refer to. 
# or cross reference. 

In [ ]:
# new thing to do...

In [ ]:
# so, in theory, each one of the cells in this output matrix needs to be compared to every cell in word_byte_array.

In [ ]:
# this is level 3, how do we get to level 4? This is no ready built in tracking? Or is there...?

In [ ]:
# # code to reference
# # levels 1 and 2
# w1b, w2b, l2 = row

# # indexer for l3
# positional_idx_l3 = (word_byte_array & l2) == 0

# # l3 words with different letters
# output_array_w3b = word_byte_array[positional_idx_l3]    

# # l3 accumulated letters
# output_array_l3 = output_array_w3b | l2

# BUILD LEVELS 3 THROUGH 5

In [ ]:
# so, now, let's try computing all possible pairs
total_output = np.full(shape = (1000000, 9), fill_value= -1, dtype = np.int32)
row_index = 0
for i_row, row in l2_df.iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row

    # indexer for l3
    positional_idx_l3 = (word_byte_array & l2) == 0

    # l3 words with different letters
    output_array_w3b = word_byte_array[positional_idx_l3]    

    # l3 accumulated letters
    output_array_l3 = output_array_w3b | l2

    ## enumerate level 3
    for w3b, l3 in zip(output_array_w3b, output_array_l3):

        # build level 4

        # l4 idx
        positional_idx_l4 = (word_byte_array & l3) == 0

        # words with different letters
        output_array_w4b = word_byte_array[positional_idx_l4]    
        
        # accumulated letters
        output_array_l4 = output_array_w4b | l3

        ## enumerate level 5
        for w4b, l4 in zip(output_array_w4b, output_array_l4):

            # build level 5

            # l5 idx
            positional_idx_l5 = (word_byte_array & l4) == 0
            
            # words with different letters
            output_array_w5b = word_byte_array[positional_idx_l5]    

            if output_array_w5b.size > 0:
                    
                # accumulated letters
                output_array_l5 = output_array_w5b | l4

                ## gather and combine the output
                for w5b, l5 in zip(output_array_w5b, output_array_l5):

                    temp_list = np.array([w1b, w2b, w3b, w4b, w5b, l2, l3, l4, l5], dtype = np.int32)
                    total_output[row_index, :] = temp_list                   
                    row_index += 1


    if i_row % 10000 == 0:
        # print the number of l2 iterations and the shape of the output
        print(i_row, row_index)
    


# CREATE AND SAVE OUTPUT

In [ ]:
total_output = total_output[:row_index, :]
col_names = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b', 'l2', 'l3', 'l4', 'l5']
l5_df = pd.DataFrame(data = total_output, columns = col_names)


In [ ]:
l5_df.shape

In [ ]:
l5_df.head()

In [ ]:
l5_df.tail()

In [ ]:
l5_df.to_csv(path_or_buf='l5.txt', sep = '\t', index = False)
